In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
ANNOTATIONS = ROOT / "data" / "annotations"
VIEWS = PROCESSED / "perspective_views"
RESULTS = ROOT / "results" / "vlm"

RESULTS.mkdir(parents=True, exist_ok=True)

images = pd.read_csv(
    PROCESSED / "perspective_views_metadata.csv"
)

reference = pd.read_csv(
    ANNOTATIONS / "human_annotations.csv",
    keep_default_na=False
)

assert len(images) == 100
assert len(reference) == 40
assert set(reference["image_id"]).issubset(set(images["image_id"]))

print("Images for inference:", len(images))
print("Images with human labels:", len(reference))
print("All image files available:",
      all((VIEWS / f).exists() for f in images["filename"]))

In [ ]:
%pip install -U python-dotenv openai pydantic

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load the .env file from the project root
load_dotenv(ROOT / ".env")

# Verify that the key was loaded
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Check your .env file."
    )

client = OpenAI(api_key=api_key)

MODEL = "gpt-4.1-mini-2025-04-14"

print("API key loaded successfully.")
print("Model:", MODEL)

In [ ]:
from typing import Literal
from pydantic import BaseModel

class StreetscapeLabels(BaseModel):
    greenery: Literal["low", "medium", "high", "unclear"]
    pedestrian: Literal["weak", "moderate", "strong", "unclear"]
    cycling: Literal["absent", "present", "unclear"]
    motor_vehicle: Literal["low", "medium", "high", "unclear"]

PROMPT = """
Classify the visible streetscape in this single image.

Use only visual evidence. Do not infer health, safety or
accessibility outcomes. Do not assume that infrastructure
exists outside the image.

Visible greenery:
low = little or no visible vegetation
medium = noticeable vegetation, not visually dominant
high = substantial vegetation or prominent tree canopy

Pedestrian infrastructure:
weak = no clearly visible dedicated pedestrian space,
       or obvious discontinuity
moderate = some identifiable pedestrian infrastructure,
           but limited or fragmented
strong = clearly visible, substantial and apparently
         continuous pedestrian space

Cycling infrastructure:
absent = no identifiable dedicated or designated facility
present = visible cycle lane, cycle track or another
          clearly designated cycling facility

Motor-vehicle dominance:
low = little visible space allocated to motor vehicles
medium = substantial motor-vehicle space alongside other uses
high = wide carriageways, extensive parking or other
       motor-vehicle infrastructure dominate the scene

Use unclear when the image does not provide sufficient
evidence for an indicator.

Return only the four requested labels.
"""

In [ ]:
import base64
import json

def classify_image(image_path):
    encoded = base64.b64encode(
        image_path.read_bytes()
    ).decode("utf-8")

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Classify this streetscape image."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{encoded}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        response_format=StreetscapeLabels,
        temperature=0
    )

    labels = response.choices[0].message.parsed

    if labels is None:
        raise ValueError("No valid structured prediction returned.")

    return labels.model_dump(), response.usage


test_results = []

for _, row in images.head(5).iterrows():

    prediction, usage = classify_image(
        VIEWS / row["filename"]
    )

    test_results.append({
        "image_id": row["image_id"],
        **prediction,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens
    })

test_df = pd.DataFrame(test_results)
display(test_df)

In [ ]:
INPUT_PRICE = 0.40 / 1_000_000
OUTPUT_PRICE = 1.60 / 1_000_000

test_cost = (
    test_df["input_tokens"].sum() * INPUT_PRICE
    + test_df["output_tokens"].sum() * OUTPUT_PRICE
)

print(f"Five-image cost: ${test_cost:.4f}")
print(f"Projected 100-image cost: ${test_cost * 20:.4f}")

test_df.to_csv(
    RESULTS / "vlm_five_image_test.csv",
    index=False
)

In [ ]:
import time
import pandas as pd

OUTPUT_FILE = RESULTS / "vlm_predictions.csv"
TEST_FILE = RESULTS / "vlm_five_image_test.csv"

# Resume an existing run, or initialize from the five-image test.
if OUTPUT_FILE.exists():
    predictions = pd.read_csv(OUTPUT_FILE)
elif TEST_FILE.exists():
    predictions = pd.read_csv(TEST_FILE)
else:
    predictions = pd.DataFrame(columns=[
        "image_id", "greenery", "pedestrian",
        "cycling", "motor_vehicle",
        "input_tokens", "output_tokens"
    ])

predictions = predictions.drop_duplicates(
    subset="image_id", keep="last"
)

completed = set(predictions["image_id"])
remaining = images[
    ~images["image_id"].isin(completed)
]

print(f"Already completed: {len(completed)}")
print(f"Remaining: {len(remaining)}")

for _, row in remaining.iterrows():
    try:
        labels, usage = classify_image(
            VIEWS / row["filename"]
        )

        record = {
            "image_id": row["image_id"],
            **labels,
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }

        predictions = pd.concat(
            [predictions, pd.DataFrame([record])],
            ignore_index=True
        )

        # Save after every successful API call.
        predictions.to_csv(OUTPUT_FILE, index=False)

        print(
            f"{len(predictions)}/100: "
            f"{row['image_id']} completed"
        )

        time.sleep(0.2)

    except Exception as e:
        print(f"Stopped at {row['image_id']}: {e}")
        break

print(f"\nSaved predictions: {len(predictions)}")

In [ ]:
predictions = pd.read_csv(OUTPUT_FILE)

assert len(predictions) == 100
assert predictions["image_id"].is_unique
assert set(predictions["image_id"]) == set(images["image_id"])

total_cost = (
    predictions["input_tokens"].sum() * INPUT_PRICE
    + predictions["output_tokens"].sum() * OUTPUT_PRICE
)

print("Total predictions:", len(predictions))
print(f"Estimated API cost: ${total_cost:.4f}")

for field in [
    "greenery", "pedestrian",
    "cycling", "motor_vehicle"
]:
    print(f"\n{field.upper()}")
    print(predictions[field].value_counts())

***Evidence-first classification***

In [ ]:
from pydantic import BaseModel
from typing import Literal

class EvidenceFirstLabels(BaseModel):
    greenery_evidence: str
    greenery: Literal["low", "medium", "high", "unclear"]

    pedestrian_evidence: str
    pedestrian: Literal["weak", "moderate", "strong", "unclear"]

    cycling_evidence: str
    cycling: Literal["absent", "present", "unclear"]

    motor_vehicle_evidence: str
    motor_vehicle: Literal["low", "medium", "high", "unclear"]


EVIDENCE_FIRST_PROMPT = """
Assess the visible streetscape using only the image.

For each indicator:
1. Identify the specific visual evidence.
2. Assign a label supported by that evidence.

GREENERY:
low: little or no visible vegetation.
medium: noticeable vegetation, not dominant.
high: substantial vegetation or prominent canopy.

PEDESTRIAN INFRASTRUCTURE:
weak: no clearly visible dedicated pedestrian space
      or obvious discontinuity.
moderate: identifiable but limited or fragmented
          pedestrian infrastructure.
strong: substantial, apparently continuous
        pedestrian space.

CYCLING INFRASTRUCTURE:
absent: no identifiable designated cycling facility.
present: a clearly identifiable designated cycle lane,
         track or other cycling facility.
unclear: evidence is insufficient or contradictory.

Parked bicycles and cyclists alone do not establish
cycling infrastructure. A bicycle parking area is not
a cycle lane or track.

Distinguish designated infrastructure from temporary
obstruction. If construction prevents you from
determining whether a cycling facility exists,
use unclear and explain why.

MOTOR-VEHICLE DOMINANCE:
low: little visible space allocated to motor vehicles.
medium: substantial vehicle space alongside other uses.
high: wide carriageways, extensive parking or other
      vehicle infrastructure dominate the scene.

Base each label on the visible evidence, not assumptions
about the surrounding neighborhood.
"""

In [ ]:
import base64
import json

def classify_evidence_first(image_path):
    encoded = base64.b64encode(
        image_path.read_bytes()
    ).decode("utf-8")

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": EVIDENCE_FIRST_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Assess this streetscape."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{encoded}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        response_format=EvidenceFirstLabels,
        temperature=0
    )

    result = response.choices[0].message.parsed

    if result is None:
        raise ValueError("No valid structured response.")

    return result.model_dump(), response.usage

In [ ]:
split_file = (
    ROOT / "results" / "evaluation"
    / "human_vs_model_with_split.csv"
)

split_df = pd.read_csv(split_file)

dev_images = (
    split_df[split_df["split"] == "development"]
    [["image_id"]]
    .merge(
        images[["image_id", "filename"]],
        on="image_id",
        validate="one_to_one"
    )
)

output_file = RESULTS / "evidence_first_development.csv"

if output_file.exists():
    revised = pd.read_csv(output_file)
else:
    revised = pd.DataFrame()

completed = (
    set(revised["image_id"])
    if not revised.empty else set()
)

for _, row in dev_images.iterrows():
    if row["image_id"] in completed:
        continue

    result, usage = classify_evidence_first(
        VIEWS / row["filename"]
    )

    record = {
        "image_id": row["image_id"],
        **result,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens
    }

    revised = pd.concat(
        [revised, pd.DataFrame([record])],
        ignore_index=True
    )

    revised.to_csv(output_file, index=False)

    print(row["image_id"], result["cycling"])

print("Completed:", len(revised))

In [ ]:
holdout = pd.read_csv(
    ANNOTATIONS / "human_annotations_holdout_reviewed.csv"
)

holdout_images = holdout[["image_id"]].merge(
    images[["image_id", "filename"]],
    on="image_id",
    validate="one_to_one"
)

assert len(holdout_images) == 24
assert holdout["location_id"].nunique() == 12

output_file = RESULTS / "evidence_first_holdout.csv"

if output_file.exists():
    holdout_predictions = pd.read_csv(output_file)
else:
    holdout_predictions = pd.DataFrame()

completed = set(
    holdout_predictions["image_id"]
) if not holdout_predictions.empty else set()

for _, row in holdout_images.iterrows():
    if row["image_id"] in completed:
        continue

    result, usage = classify_evidence_first(
        VIEWS / row["filename"]
    )

    record = {
        "image_id": row["image_id"],
        **result,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens
    }

    holdout_predictions = pd.concat(
        [holdout_predictions, pd.DataFrame([record])],
        ignore_index=True
    )

    holdout_predictions.to_csv(output_file, index=False)
    completed.add(row["image_id"])

    print(row["image_id"], result["cycling"])

assert len(holdout_predictions) == 24
print("Holdout inference complete.")

In [ ]:
dev = pd.read_csv(
    RESULTS / "evidence_first_development.csv"
)

holdout = pd.read_csv(
    RESULTS / "evidence_first_holdout.csv"
)

output_file = RESULTS / "evidence_first_all.csv"

if output_file.exists():
    evidence_all = pd.read_csv(output_file)
else:
    evidence_all = pd.concat(
        [dev, holdout],
        ignore_index=True
    )

evidence_all = evidence_all.drop_duplicates(
    subset="image_id",
    keep="last"
)

evidence_all.to_csv(output_file, index=False)

completed = set(evidence_all["image_id"])

remaining = images[
    ~images["image_id"].isin(completed)
]

print(f"Already completed: {len(completed)}")
print(f"Remaining: {len(remaining)}")

for _, row in remaining.iterrows():
    try:
        result, usage = classify_evidence_first(
            VIEWS / row["filename"]
        )

        record = {
            "image_id": row["image_id"],
            **result,
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }

        evidence_all = pd.concat(
            [evidence_all, pd.DataFrame([record])],
            ignore_index=True
        )

        evidence_all.to_csv(output_file, index=False)

        print(
            f"{len(evidence_all)}/100: "
            f"{row['image_id']}"
        )

    except Exception as e:
        print(f"Stopped: {e}")
        break

assert len(evidence_all) == 100
assert evidence_all["image_id"].is_unique